# Regression models for enantioselectivity of the asymmetric Mannich reaction

Notebook 2 of 2. Trains and evaluates regression models for |ΔΔG‡| (kcal/mol) on the
representations computed in notebook 1 (`mannich_representations.ipynb`).

**Inputs** (one folder, set in the configuration cell)
- `mannich_dataset_hbd.csv` — reactions, target, grouping columns,
  atom-mapped reaction SMILES, catalyst SMILES, axial configuration.
- `representations/features_*.csv` — feature matrices keyed by `reaction_id`
  (physchem, MACCS, ECFP4, ECFP4-count, DRFP, one-hot; DFT optional) and `features_conditions.csv`.

**Models**: Dummy (mean), Ridge, SVR (RBF), Gaussian process, Random Forest, XGBoost, chemprop v2 D-MPNN.

**Validation schemes**: random 5-fold × 5 repeats; leave-one-group-out by `research_group`;
GroupKFold (10 folds) by `catalyst_id`.

**Hyperparameters**: tuned once per split scheme, representation and model on the training part of the
first outer fold, then fixed for all outer folds (`HP_MODE = "once"`); full nested search per fold is
available as `HP_MODE = "per_fold"`.

**Outputs** (`results/`): out-of-fold predictions, metric summary (MAE, RMSE, R², Spearman ρ,
Kendall τ) with cluster-bootstrap 95% CIs,
fold assignments, selected hyperparameters, parity plots, summary heatmap, `manifest.json`.

Sections: 1 Configuration · 2 Environment · 3 Data · 4 Splits · 5 Models · 6 Classical models ·
7 chemprop · 8 Results · 9 Metrics · 10 Figures · 11 Manifest

## 1. Configuration
All paths and switches. Nothing below this cell needs editing.

In [ ]:
import os
from pathlib import Path

# --- BLAS threads (must be set before numpy is imported) ------------------------------
# Multithreaded OpenBLAS crashed the kernel (SIGBUS) on this machine; one BLAS thread avoids it.
# RF / XGBoost parallelism (N_JOBS) is not affected.
BLAS_THREADS = "1"
for _var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS_BLAS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_var, BLAS_THREADS)

# --- Paths ---------------------------------------------------------------------------
# Paths are relative to notebooks/ (run the notebook from that folder).
DATA_DIR = Path("../data")          # processed/ (dataset) and representations/
DATASET_FILE = "mannich_dataset_hbd.csv"
REPR_DIR = DATA_DIR / "representations"
FOLDS_DIR = Path("../splits")       # fold assignments, shared with the classification notebook
RESULTS_DIR = Path("../results/regression")

# --- What to run ---------------------------------------------------------------------
RUN_REPRESENTATIONS = ["physchem", "maccs", "ecfp4", "ecfp4_count", "drfp", "onehot", "dft"]
RUN_MODELS = ["dummy", "ridge", "svr", "gp", "rf", "xgb", "chemprop"]
RUN_SPLITS = ["random", "lorgo_group", "gkf10_catalyst"]

# Optional override from the command line: MANNICH_RUN_MODELS=classical | chemprop
# (runs the classical models and chemprop in separate processes; results meet in the shared cache)
_run_models = os.environ.get("MANNICH_RUN_MODELS", "").strip().lower()
if _run_models == "classical":
    RUN_MODELS = [m for m in RUN_MODELS if m != "chemprop"]
elif _run_models == "chemprop":
    RUN_MODELS = ["chemprop"]
elif _run_models:
    raise ValueError(f"MANNICH_RUN_MODELS must be 'classical' or 'chemprop', got {_run_models!r}")

# --- Cross-validation ----------------------------------------------------------------
SEED = 0                               # base seed (estimators, searches, bootstrap)
RANDOM_REPEAT_SEEDS = [0, 1, 2, 3, 4]  # KFold shuffling seeds, one per repeat
N_FOLDS_RANDOM = 5
N_FOLDS_GKF_CATALYST = 10
N_INNER_FOLDS = 5                      # inner CV for hyperparameter search
N_ITER = 20                            # RandomizedSearchCV candidates per search
HP_MODE = "once"                       # "once": tune on the training part of the first outer fold,
                                       #         then fixed for all folds of that split
                                       # "per_fold": nested search inside every outer fold (slow)
N_BOOT = 1000                          # cluster-bootstrap replicates
CI_LEVEL = 0.95

# --- Compute -------------------------------------------------------------------------
N_JOBS = -1          # threads for RF / XGBoost; hyperparameter searches run in one process
GP_N_RESTARTS = 2    # restarts of the marginal-likelihood optimiser

# --- chemprop (fixed hyperparameters, no nested search) -------------------------------
CHEMPROP_DEVICE = "cpu"   # "cpu", "mps" (Apple Silicon) or "auto" (mps if available, else cpu)
CHEMPROP = dict(
    message_hidden_dim=300, depth=3, dropout=0.1,
    ffn_hidden_dim=300, ffn_num_layers=2,
    batch_size=64, max_epochs=150, patience=20,
    init_lr=1e-4, max_lr=1e-3, final_lr=1e-4, warmup_epochs=2,
    val_fraction=0.1,      # internal validation part for early stopping
)

## 2. Environment check

In [ ]:
import importlib
import platform
import sys


from importlib import metadata

DISTRIBUTIONS = {"sklearn": "scikit-learn"}


def package_version(name):
    """Installed version from package metadata; the package itself is not imported."""
    try:
        return metadata.version(DISTRIBUTIONS.get(name, name))
    except metadata.PackageNotFoundError:
        return None


PACKAGES = ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "xgboost",
            "torch", "lightning", "chemprop", "rdkit"]
VERSIONS = {p: package_version(p) for p in PACKAGES}

print(f"Python {sys.version.split()[0]} | {platform.platform()}")
for p, v in VERSIONS.items():
    print(f"  {p:<11} {v or 'not installed'}")

required = {"numpy", "pandas", "scipy", "sklearn", "matplotlib"}
if "xgb" in RUN_MODELS:
    required.add("xgboost")
if "chemprop" in RUN_MODELS:
    required |= {"chemprop", "torch", "lightning", "rdkit"}
missing = sorted(p for p in required if VERSIONS[p] is None)
if missing:
    raise ImportError(f"Missing packages for the current RUN_MODELS: {missing}")

if "chemprop" in RUN_MODELS:
    if int(VERSIONS["chemprop"].split(".")[0]) < 2:
        raise ImportError("chemprop >= 2.0 is required")
    import torch
    print(f"  torch backends: MPS={torch.backends.mps.is_available()}, CUDA={torch.cuda.is_available()}")
if "chemprop" in RUN_MODELS and len(RUN_MODELS) > 1:
    print("  note: chemprop (PyTorch) and the classical models (XGBoost) share one process in this run; "
          "on macOS run them separately (MANNICH_RUN_MODELS=classical / chemprop) to avoid OpenMP conflicts")

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # CPU fallback for ops missing on MPS

import gc
import hashlib
import json
import logging
import pickle
import time
import warnings
from dataclasses import replace
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize, TwoSlopeNorm
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.stats import kendalltau, loguniform, randint, spearmanr, uniform
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (ConstantKernel, Hyperparameter, Kernel, Matern,
                                              NormalizedKernelMixin, StationaryKernelMixin,
                                              WhiteKernel)
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, KFold, RandomizedSearchCV,
                                     ShuffleSplit)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

if "xgb" in RUN_MODELS:
    from xgboost import XGBRegressor

CODE_VERSION = "1"   # bump to invalidate cached fold results after code changes
CACHE_DIR = RESULTS_DIR / "cache"
FIG_DIR = RESULTS_DIR / "figures"
for d in (RESULTS_DIR, CACHE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
LOG_FILE = RESULTS_DIR / "run_log.txt"


def memory_mb():
    """Resident memory of this process (MB); peak RSS if psutil is unavailable."""
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2**20
    except Exception:
        import resource
        peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak / 2**20 if sys.platform == "darwin" else peak / 2**10


def log(message):
    """Print and append a time-stamped line to results/run_log.txt (progress survives a lost front end)."""
    print(message, flush=True)
    with open(LOG_FILE, "a") as fh:
        fh.write(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  {message}\n")


def release_memory():
    """Free memory between representations: GP kernel cache, joblib worker processes, garbage."""
    try:
        _SimilarityKernel._CACHE.clear()
    except NameError:
        pass
    try:
        from joblib.externals.loky import get_reusable_executor
        get_reusable_executor().shutdown(wait=True)
    except Exception:
        pass
    gc.collect()


import faulthandler
_fault_log = open(RESULTS_DIR / "faulthandler.log", "a")
_fault_log.write(f"\n=== session {datetime.now().isoformat(timespec='seconds')}\n")
_fault_log.flush()
faulthandler.enable(file=_fault_log, all_threads=True)   # Python traceback of a hard crash goes here

try:   # enforce the BLAS thread limit at run time as well (environment variables may be read too late)
    from threadpoolctl import threadpool_limits
    _blas_limit = threadpool_limits(limits=int(BLAS_THREADS), user_api="blas")
except Exception:
    pass

log(f"=== session start | RUN_MODELS={RUN_MODELS} | HP_MODE={HP_MODE} | torch loaded: {'torch' in sys.modules} "
    f"| xgboost loaded: {'xgboost' in sys.modules} | memory {memory_mb():.0f} MB")

plt.rcParams.update({"font.size": 8, "pdf.fonttype": 42, "ps.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})

## 3. Data

Rows of every representation file must carry the same `reaction_id` values in the same order as the
dataset; a mismatch stops the notebook. Each representation is merged with `features_conditions.csv`
(temperature, catalyst loading, Kamlet–Taft α, β, π*, dielectric constant) on `reaction_id`.

Representation types define fold-internal preprocessing and the GP kernel:
- *continuous* (physchem, DFT): all columns standardised; GP with Matérn (ν = 2.5) kernel;
- *binary* (MACCS, ECFP4, DRFP, one-hot) and *count* (ECFP4-count): the fingerprint block is not scaled;
  conditions are standardised; GP with Tanimoto (binary) or MinMax (count) kernel on the fingerprint block
  plus an RBF kernel on the conditions.

Catalyst blocks contain the flags `cat_axial_R` / `cat_axial_S`; where present, they are checked against
`axial_configuration` of the dataset.

In [ ]:
REPRESENTATIONS = {   # name: (file, type)
    "physchem":    ("features_physchem.csv", "continuous"),
    "maccs":       ("features_maccs.csv", "binary"),
    "ecfp4":       ("features_ecfp4.csv", "binary"),
    "ecfp4_count": ("features_ecfp4_count.csv", "count"),
    "drfp":        ("features_drfp.csv", "binary"),
    "onehot":      ("features_onehot.csv", "binary"),
    "dft":         ("features_dft.csv", "continuous"),
}
OPTIONAL_REPRESENTATIONS = {"dft"}
CONDITIONS_FILE = "features_conditions.csv"

REP_LABELS = {"physchem": "Physchem", "maccs": "MACCS", "ecfp4": "ECFP4", "ecfp4_count": "ECFP4-count",
              "drfp": "DRFP", "onehot": "OHE", "dft": "DFT", "cgr": "CGR + catalyst"}
MODEL_LABELS = {"dummy": "Dummy", "ridge": "Ridge", "svr": "SVR", "gp": "GP", "rf": "RF",
                "xgb": "XGBoost", "chemprop": "D-MPNN"}
REP_ORDER = list(REPRESENTATIONS) + ["cgr"]
MODEL_ORDER = list(MODEL_LABELS)

unknown = (set(RUN_REPRESENTATIONS) - set(REPRESENTATIONS)) | (set(RUN_MODELS) - set(MODEL_LABELS))
if unknown:
    raise ValueError(f"Unknown names in RUN_REPRESENTATIONS / RUN_MODELS: {sorted(unknown)}")


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


# --- dataset -------------------------------------------------------------------------
DATASET_PATH = DATA_DIR / "processed" / DATASET_FILE
REQUIRED_COLUMNS = ["reaction_id", "ddG_kcal_mol", "research_group", "catalyst_id", "paper_doi",
                    "mapped_rxn", "organocatalyst", "axial_configuration"]
data = pd.read_csv(DATASET_PATH)
missing_cols = [c for c in REQUIRED_COLUMNS if c not in data.columns]
if missing_cols:
    raise ValueError(f"Dataset lacks columns: {missing_cols}")
if data["reaction_id"].duplicated().any():
    raise ValueError("Duplicated reaction_id in the dataset")
for c in ["ddG_kcal_mol", "research_group", "catalyst_id"]:
    if data[c].isna().any():
        raise ValueError(f"Missing values in {c}")

REACTION_IDS = data["reaction_id"].to_numpy()
POS = pd.Series(np.arange(len(data)), index=REACTION_IDS)   # reaction_id -> row position
Y = data["ddG_kcal_mol"].to_numpy(float)
GROUPS = {"research_group": data["research_group"].astype(str).to_numpy(),
          "catalyst_id": data["catalyst_id"].astype(str).to_numpy()}

axial = data["axial_configuration"].fillna("none").astype(str).str.strip()
axial = axial.map(lambda s: {"r": "R", "s": "S", "none": "none"}.get(s.lower(), s))
bad_axial = sorted(set(axial) - {"R", "S", "none"})
if bad_axial:
    raise ValueError(f"Unexpected axial_configuration values: {bad_axial}")
AXIAL_FLAGS = np.column_stack([axial == "R", axial == "S"]).astype(float)

cat_counts = data["catalyst_id"].value_counts()
print(f"Reactions: {len(data)}")
print(f"research_group: {data['research_group'].nunique()} | paper_doi: {data['paper_doi'].nunique()} | "
      f"catalyst_id: {len(cat_counts)} ({(cat_counts == 1).sum()} with a single reaction)")
print(f"axial_configuration: {axial.value_counts().to_dict()}")
print(f"|ddG| (kcal/mol): mean {Y.mean():.3f}, sd {Y.std(ddof=1):.3f}, "
      f"min {Y.min():.3f}, max {Y.max():.3f}")


def check_reaction_ids(frame, label):
    """Same reaction_id values, in the same order, as the dataset."""
    if frame.columns[0] != "reaction_id":
        raise ValueError(f"{label}: first column must be 'reaction_id', found {frame.columns[0]!r}")
    ids = frame["reaction_id"].to_numpy()
    if len(ids) != len(REACTION_IDS):
        raise ValueError(f"{label}: {len(ids)} rows, dataset has {len(REACTION_IDS)}")
    if not np.array_equal(ids, REACTION_IDS):
        what = "order" if set(ids) == set(REACTION_IDS) else "set"
        raise ValueError(f"{label}: reaction_id {what} differs from the dataset")


# --- conditions ----------------------------------------------------------------------
conditions = pd.read_csv(REPR_DIR / CONDITIONS_FILE)
check_reaction_ids(conditions, CONDITIONS_FILE)
COND_COLS = [c for c in conditions.columns if c != "reaction_id"]
print(f"Conditions ({len(COND_COLS)}): {COND_COLS}; "
      f"missing values: {int(conditions[COND_COLS].isna().sum().sum())}")

# --- input hashes (manifest, cache signatures) ---------------------------------------
FILE_HASHES = {DATASET_FILE: sha256(DATASET_PATH), CONDITIONS_FILE: sha256(REPR_DIR / CONDITIONS_FILE)}
for name, (fname, _) in REPRESENTATIONS.items():
    if (REPR_DIR / fname).exists():
        FILE_HASHES[fname] = sha256(REPR_DIR / fname)
    elif name not in OPTIONAL_REPRESENTATIONS and name in RUN_REPRESENTATIONS:
        raise FileNotFoundError(REPR_DIR / fname)
print("Representation files found:",
      [n for n, (f, _) in REPRESENTATIONS.items() if f in FILE_HASHES])

In [ ]:
def load_representation(name):
    """Representation block merged with conditions. Returns None if an optional file is absent.

    X columns are ordered [representation | conditions]; n_fp is the size of the fingerprint
    block (0 for continuous representations, where all columns are treated as continuous).
    """
    fname, kind = REPRESENTATIONS[name]
    path = REPR_DIR / fname
    if not path.exists():
        if name in OPTIONAL_REPRESENTATIONS:
            return None
        raise FileNotFoundError(path)
    rep = pd.read_csv(path)
    check_reaction_ids(rep, fname)
    rep_cols = [c for c in rep.columns if c != "reaction_id"]
    overlap = set(rep_cols) & set(COND_COLS)
    if overlap:
        print(f"  {fname}: columns also in conditions, kept once from conditions: {sorted(overlap)}")
        rep_cols = [c for c in rep_cols if c not in overlap]
    non_numeric = [c for c in rep_cols if not pd.api.types.is_numeric_dtype(rep[c])]
    if non_numeric:
        raise ValueError(f"{fname}: non-numeric columns {non_numeric[:5]}")

    merged = rep[["reaction_id"] + rep_cols].merge(conditions, on="reaction_id", how="inner",
                                                    validate="one_to_one")
    check_reaction_ids(merged, f"{fname} + {CONDITIONS_FILE}")

    Xr = merged[rep_cols].to_numpy(float)
    values = Xr[~np.isnan(Xr)]
    if kind == "binary" and not np.isin(values, (0.0, 1.0)).all():
        raise ValueError(f"{fname}: declared binary but contains values other than 0/1")
    if kind == "count" and ((values < 0).any() or (values != np.round(values)).any()):
        raise ValueError(f"{fname}: declared count but contains negative or non-integer values")

    if {"cat_axial_R", "cat_axial_S"} <= set(rep_cols):
        flags = merged[["cat_axial_R", "cat_axial_S"]].to_numpy(float)
        if not np.array_equal(flags, AXIAL_FLAGS):
            raise ValueError(f"{fname}: cat_axial_R/S disagree with axial_configuration")

    X = np.hstack([Xr, merged[COND_COLS].to_numpy(float)])
    n_fp = 0 if kind == "continuous" else len(rep_cols)
    return {"name": name, "kind": kind, "X": X, "n_fp": n_fp, "columns": rep_cols + COND_COLS,
            "file": fname}

## 4. Splits

| Split | Outer folds | Inner folds (search) | Bootstrap cluster |
|---|---|---|---|
| `random` | KFold 5 × 5 repeats (seeds 0–4) | KFold 5 | reaction |
| `lorgo_group` | leave-one-group-out, `research_group` (43 folds) | GroupKFold 5, `research_group` | `research_group` |
| `gkf10_catalyst` | GroupKFold 10, `catalyst_id` | GroupKFold 5, `catalyst_id` | `catalyst_id` |

All models, including chemprop, use the same folds. Repeated reactions are not grouped in the random
split; random-CV scores therefore include interpolation between replicate measurements. Catalyst-wise
generalisation is assessed with GroupKFold rather than leave-one-catalyst-out, as in repeated
catalyst-grouped CV used for literature enantioselectivity data (e.g. Hadler et al., JACS 2026).
Fold assignments are written to `folds_<split>.csv`
on the first run and read back afterwards, so all models and later runs use identical folds.

In [ ]:
ALL_SPLITS = ["random", "lorgo_group", "gkf10_catalyst"]
CHEMPROP_SPLITS = ALL_SPLITS
SPLIT_GROUP = {"random": None, "lorgo_group": "research_group", "gkf10_catalyst": "catalyst_id"}
SPLIT_LABELS = {"random": "Random (5-fold × 5)", "lorgo_group": "LORGO (research group)",
                "gkf10_catalyst": "GroupKFold-10 (catalyst)"}
if HP_MODE not in ("once", "per_fold"):
    raise ValueError(f"HP_MODE must be 'once' or 'per_fold', got {HP_MODE!r}")
if set(RUN_SPLITS) - set(ALL_SPLITS):
    raise ValueError(f"Unknown splits: {sorted(set(RUN_SPLITS) - set(ALL_SPLITS))}")


def make_folds(split):
    n = len(REACTION_IDS)
    if split == "random":
        parts = []
        for repeat, seed in enumerate(RANDOM_REPEAT_SEEDS):
            fold = np.empty(n, dtype=int)
            for k, (_, te) in enumerate(KFold(N_FOLDS_RANDOM, shuffle=True, random_state=seed).split(np.zeros(n))):
                fold[te] = k
            parts.append(pd.DataFrame({"reaction_id": REACTION_IDS, "repeat": repeat, "fold": fold,
                                       "group": ""}))
        return pd.concat(parts, ignore_index=True)
    g = GROUPS[SPLIT_GROUP[split]]
    if split == "lorgo_group":
        fold = np.searchsorted(np.unique(g), g)
    else:
        fold = np.empty(n, dtype=int)
        for k, (_, te) in enumerate(GroupKFold(N_FOLDS_GKF_CATALYST).split(np.zeros(n), groups=g)):
            fold[te] = k
    return pd.DataFrame({"reaction_id": REACTION_IDS, "repeat": 0, "fold": fold, "group": g})


def validate_folds(folds, split):
    for repeat, sub in folds.groupby("repeat"):
        if len(sub) != len(REACTION_IDS) or set(sub["reaction_id"]) != set(REACTION_IDS):
            raise ValueError(f"folds_{split}.csv, repeat {repeat}: does not cover each reaction once")
    if split != "random":
        g = pd.Series(GROUPS[SPLIT_GROUP[split]], index=REACTION_IDS)
        if not (folds["group"].astype(str).to_numpy() == g.loc[folds["reaction_id"]].to_numpy()).all():
            raise ValueError(f"folds_{split}.csv: group labels differ from the dataset; delete the file to regenerate")
        if (folds.groupby("group")["fold"].nunique() > 1).any():
            raise ValueError(f"folds_{split}.csv: a group is spread over several folds")


FOLDS, FOLD_HASHES = {}, {}
for split in ALL_SPLITS:
    path = FOLDS_DIR / f"folds_{split}.csv"
    if path.exists():
        folds = pd.read_csv(path, dtype={"group": str}, keep_default_na=False)
        status = "loaded"
    else:
        folds = make_folds(split)
        folds.insert(1, "split", split)
        folds.to_csv(path, index=False)
        status = "created"
    validate_folds(folds, split)
    FOLDS[split] = folds
    FOLD_HASHES[split] = sha256(path)
    sizes = folds.groupby(["repeat", "fold"]).size()
    print(f"{split:<15} {status:<7} repeats={folds['repeat'].nunique()} folds/repeat={folds['fold'].nunique():>3} "
          f"test size min/median/max = {sizes.min()}/{int(sizes.median())}/{sizes.max()}")


def iter_folds(split):
    """Yields (repeat, fold, train_positions, test_positions)."""
    folds = FOLDS[split]
    for repeat, rep_folds in folds.groupby("repeat", sort=True):
        for fold in sorted(rep_folds["fold"].unique()):
            is_test = (rep_folds["fold"] == fold).to_numpy()
            ids = rep_folds["reaction_id"].to_numpy()
            yield int(repeat), int(fold), POS.loc[ids[~is_test]].to_numpy(), POS.loc[ids[is_test]].to_numpy()


def n_fold_units(split):
    return FOLDS[split].groupby(["repeat", "fold"]).ngroups

## 5. Models

**Fold-internal preprocessing** (`BlockPreprocessor`, fitted on training rows only): median imputation,
removal of columns constant in the training rows, standardisation of continuous columns (Ridge, SVR, GP;
not for trees). For one-hot encoding, a component absent from the training rows gives an all-zero column
that is removed, so a test reaction with that component is encoded by a zero block for it — this is the
expected behaviour of one-hot features for unseen components.

**Hyperparameters** (Ridge, SVR, RF, XGBoost): `RandomizedSearchCV` with `N_ITER` candidates, MAE criterion and
inner folds that follow the outer logic (KFold / GroupKFold by `research_group` / GroupKFold by `catalyst_id`).
- `HP_MODE = "once"` (default): one search per split scheme × representation × model, run on the training
  part of the first outer fold (repeat 0, fold 0); the selected values are then fixed and each outer fold
  refits the model with them. The test reactions of the first fold never enter the search; reactions in the
  test parts of the other folds take part in choosing the hyperparameters (not in fitting the evaluated
  models). This follows common practice for literature enantioselectivity models (tuning once on a training
  set with k-fold CV, e.g. Hadler et al., JACS 2026; Betinol et al., JACS 2023).
- `HP_MODE = "per_fold"`: the search is repeated inside every outer fold (full nested CV).

GP kernel hyperparameters are fitted in every fold by maximising the log marginal likelihood
(`normalize_y=True`); chemprop uses fixed hyperparameters (Section 7).

**GP kernels**
- fingerprints: `C·Tanimoto(fp) + C·RBF_ARD(conditions) + White`; for ECFP4-count the Tanimoto kernel
  is replaced by MinMax, Σmin(x,y)/Σmax(x,y);
- continuous: `C·Matérn(ν=2.5) + White` on standardised features (isotropic length scale).

In [ ]:
class BlockPreprocessor(BaseEstimator, TransformerMixin):
    """Imputation, constant-column removal and optional scaling for X = [fingerprint | continuous].

    All statistics come from the rows passed to fit(). The output keeps the block order;
    `n_fp_out_` is the number of retained fingerprint columns.
    """

    def __init__(self, n_fp=0, scale_continuous=True):
        self.n_fp = n_fp
        self.scale_continuous = scale_continuous

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)   # all-NaN columns
            med = np.nanmedian(X, axis=0)
        self.medians_ = np.where(np.isnan(med), 0.0, med)
        Xi = np.where(np.isnan(X), self.medians_, X)
        keep = Xi.max(axis=0) > Xi.min(axis=0)
        is_fp = np.arange(X.shape[1]) < self.n_fp
        self.fp_idx_ = np.flatnonzero(keep & is_fp)
        self.cont_idx_ = np.flatnonzero(keep & ~is_fp)
        self.n_fp_out_ = len(self.fp_idx_)
        if self.scale_continuous and len(self.cont_idx_):
            C = Xi[:, self.cont_idx_]
            self.mean_ = C.mean(axis=0)
            sd = C.std(axis=0)
            self.scale_ = np.where(sd > 0, sd, 1.0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        Xi = np.where(np.isnan(X), self.medians_, X)
        cont = Xi[:, self.cont_idx_]
        if self.scale_continuous and len(self.cont_idx_):
            cont = (cont - self.mean_) / self.scale_
        return np.hstack([Xi[:, self.fp_idx_], cont])


# --- GP kernels ----------------------------------------------------------------------
class _SimilarityKernel(Kernel):
    """Similarity on columns [start, stop) of X; no hyperparameters.

    K(X, X) is cached (keyed by content hash), because sklearn re-evaluates the kernel at every
    step of the marginal-likelihood optimisation while only the other kernel terms change.
    """
    _CACHE = {}

    def __init__(self, start=0, stop=None):
        self.start = start
        self.stop = stop

    def _block(self, X):
        return np.asarray(X, dtype=float)[:, self.start:self.stop]

    def _similarity(self, A, B):
        raise NotImplementedError

    def __call__(self, X, Y=None, eval_gradient=False):
        A = self._block(X)
        if Y is None:
            key = (type(self).__name__, hashlib.blake2b(np.ascontiguousarray(A).tobytes(), digest_size=16).hexdigest())
            if key not in self._CACHE:
                if len(self._CACHE) >= 4:
                    self._CACHE.clear()
                self._CACHE[key] = self._similarity(A, A)
            K = self._CACHE[key].copy()
        else:
            if eval_gradient:
                raise ValueError("Gradient can only be evaluated when Y is None.")
            K = self._similarity(A, self._block(Y))
        if eval_gradient:
            return K, np.empty((K.shape[0], K.shape[0], 0))
        return K

    def diag(self, X):
        return np.ones(np.asarray(X).shape[0])

    def is_stationary(self):
        return False

    def __repr__(self):
        return f"{type(self).__name__}(cols={self.start}:{self.stop})"


class TanimotoKernel(_SimilarityKernel):
    """<a,b> / (|a|² + |b|² − <a,b>); equals Jaccard similarity for binary vectors."""

    def _similarity(self, A, B):
        AB = (A.astype(np.float32) @ B.astype(np.float32).T).astype(np.float64)
        denom = (A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - AB
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(denom > 0, AB / denom, 1.0)


class MinMaxKernel(_SimilarityKernel):
    """Σ min(a,b) / Σ max(a,b) for non-negative integer counts.

    Σ min is computed as Σ_t <[a ≥ t], [b ≥ t]> over count thresholds t (matrix products).
    """

    def _similarity(self, A, B):
        A = np.rint(A).astype(np.int32)
        B = np.rint(B).astype(np.int32)
        col_max = np.maximum(A.max(0, initial=0), B.max(0, initial=0))
        smin = np.zeros((A.shape[0], B.shape[0]))
        for t in range(1, int(col_max.max(initial=0)) + 1):
            cols = np.flatnonzero(col_max >= t)
            smin += ((A[:, cols] >= t).astype(np.float32) @ (B[:, cols] >= t).astype(np.float32).T)
        smax = A.sum(1)[:, None] + B.sum(1)[None, :] - smin
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(smax > 0, smin / smax, 1.0)


class SlicedRBF(StationaryKernelMixin, NormalizedKernelMixin, Kernel):
    """RBF kernel (scalar or ARD length scales) on columns [start, stop) of X."""

    def __init__(self, start=0, stop=None, length_scale=1.0, length_scale_bounds=(1e-2, 1e3)):
        self.start = start
        self.stop = stop
        self.length_scale = length_scale
        self.length_scale_bounds = length_scale_bounds

    @property
    def anisotropic(self):
        return np.iterable(self.length_scale) and len(self.length_scale) > 1

    @property
    def hyperparameter_length_scale(self):
        if self.anisotropic:
            return Hyperparameter("length_scale", "numeric", self.length_scale_bounds, len(self.length_scale))
        return Hyperparameter("length_scale", "numeric", self.length_scale_bounds)

    def __call__(self, X, Y=None, eval_gradient=False):
        A = np.asarray(X, dtype=float)[:, self.start:self.stop]
        ls = np.squeeze(np.asarray(self.length_scale, dtype=float))
        if Y is None:
            D = squareform(pdist(A / ls, metric="sqeuclidean"))
        else:
            if eval_gradient:
                raise ValueError("Gradient can only be evaluated when Y is None.")
            D = cdist(A / ls, np.asarray(Y, dtype=float)[:, self.start:self.stop] / ls, metric="sqeuclidean")
        K = np.exp(-0.5 * D)
        if not eval_gradient:
            return K
        if self.hyperparameter_length_scale.fixed:
            return K, np.empty((K.shape[0], K.shape[0], 0))
        if self.anisotropic:
            G = (A[:, None, :] - A[None, :, :]) ** 2 / ls ** 2
            return K, G * K[..., None]
        return K, (K * D)[..., None]

    def __repr__(self):
        ls = ", ".join(f"{v:.3g}" for v in np.atleast_1d(self.length_scale))
        return f"RBF(cols={self.start}:{self.stop}, length_scale=[{ls}])"


class GPRegressorBlocks(BaseEstimator, RegressorMixin):
    """Gaussian process with fold-internal preprocessing and a kernel chosen by representation type."""

    def __init__(self, kind="binary", n_fp=0, n_restarts=2, random_state=0):
        self.kind = kind
        self.n_fp = n_fp
        self.n_restarts = n_restarts
        self.random_state = random_state

    def _kernel(self, n_fp, d):
        amp = lambda: ConstantKernel(1.0, (1e-3, 1e3))
        noise = WhiteKernel(0.1, (1e-5, 1e1))
        if self.kind == "continuous" or n_fp == 0:
            return amp() * Matern(length_scale=np.sqrt(d), length_scale_bounds=(1e-2, 1e4), nu=2.5) + noise
        sim = TanimotoKernel(0, n_fp) if self.kind == "binary" else MinMaxKernel(0, n_fp)
        kernel = amp() * sim
        if d > n_fp:
            kernel = kernel + amp() * SlicedRBF(n_fp, d, length_scale=np.ones(d - n_fp))
        return kernel + noise

    def fit(self, X, y):
        self.prep_ = BlockPreprocessor(self.n_fp, scale_continuous=True).fit(X)
        Z = self.prep_.transform(X)
        self.gpr_ = GaussianProcessRegressor(self._kernel(self.prep_.n_fp_out_, Z.shape[1]), normalize_y=True,
                                             n_restarts_optimizer=self.n_restarts,
                                             random_state=self.random_state)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            self.gpr_.fit(Z, y)
        return self

    def predict(self, X, return_std=False):
        return self.gpr_.predict(self.prep_.transform(X), return_std=return_std)

    def fitted_params(self):
        return {"kernel": str(self.gpr_.kernel_),
                "log_marginal_likelihood": float(self.gpr_.log_marginal_likelihood_value_)}

In [ ]:
# Search spaces: declarative (for cache signatures), converted to scipy distributions below.
PARAM_SPACES = {
    "ridge": {"model__alpha": ("loguniform", 1e-3, 1e4)},
    "svr": {"model__C": ("loguniform", 1e-2, 1e3),
            "model__gamma": ("loguniform", 1e-5, 1e-1),
            "model__epsilon": ("uniform", 0.01, 0.5)},
    "rf": {"model__n_estimators": ("choice", [200, 500]),
           "model__max_depth": ("choice", [None, 8, 16, 32]),
           "model__max_features": ("choice", ["sqrt", 0.1, 0.3]),
           "model__min_samples_leaf": ("choice", [1, 2, 4, 8])},
    "xgb": {"model__max_depth": ("randint", 2, 8),
            "model__learning_rate": ("loguniform", 0.01, 0.3),
            "model__n_estimators": ("randint", 100, 1000),
            "model__subsample": ("uniform", 0.6, 1.0),
            "model__colsample_bytree": ("uniform", 0.3, 1.0),
            "model__reg_lambda": ("loguniform", 1e-2, 1e2)},
}


def to_distribution(spec):
    kind, *args = spec
    if kind == "loguniform":
        return loguniform(args[0], args[1])
    if kind == "uniform":                      # [low, high]
        return uniform(args[0], args[1] - args[0])
    if kind == "randint":                      # [low, high], inclusive
        return randint(args[0], args[1] + 1)
    if kind == "choice":
        return list(args[0])
    raise ValueError(kind)


def make_estimator(model, rep, seed):
    n_fp = rep["n_fp"]
    if model == "dummy":
        return DummyRegressor(strategy="mean")
    if model == "ridge":
        return Pipeline([("prep", BlockPreprocessor(n_fp, True)), ("model", Ridge())])
    if model == "svr":
        return Pipeline([("prep", BlockPreprocessor(n_fp, True)), ("model", SVR(kernel="rbf"))])
    if model == "gp":
        return GPRegressorBlocks(kind=rep["kind"], n_fp=n_fp, n_restarts=GP_N_RESTARTS, random_state=seed)
    if model == "rf":
        return Pipeline([("prep", BlockPreprocessor(n_fp, False)),
                         ("model", RandomForestRegressor(random_state=seed, n_jobs=N_JOBS))])
    if model == "xgb":
        return Pipeline([("prep", BlockPreprocessor(n_fp, False)),
                         ("model", XGBRegressor(tree_method="hist", random_state=seed, n_jobs=N_JOBS,
                                                verbosity=0))])
    raise ValueError(model)


def inner_cv(split, n_train_groups, seed):
    if split == "random":
        return KFold(N_INNER_FOLDS, shuffle=True, random_state=seed)
    return GroupKFold(min(N_INNER_FOLDS, n_train_groups))


def to_builtin(obj):
    """JSON-serialisable copy of a parameter dict."""
    if isinstance(obj, dict):
        return {k: to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def run_search(model, rep, split, train, seed):
    """RandomizedSearchCV on the rows `train`; returns (best params without the 'model__' prefix, inner MAE)."""
    est = make_estimator(model, rep, seed)
    groups = None if split == "random" else GROUPS[SPLIT_GROUP[split]][train]
    n_groups = len(np.unique(groups)) if groups is not None else len(train)
    space = {k: to_distribution(v) for k, v in PARAM_SPACES[model].items()}
    search = RandomizedSearchCV(est, space, n_iter=N_ITER, scoring="neg_mean_absolute_error",
                                cv=inner_cv(split, n_groups, seed), refit=False, random_state=seed,
                                n_jobs=1, error_score="raise")
    search.fit(rep["X"][train], Y[train], groups=groups)
    return search.best_params_, float(-search.best_score_)


def tuned_once(model, rep, split):
    """Search on the training part of the first outer fold (repeat 0, fold 0), cached on disk."""
    path = CACHE_DIR / split / rep["name"] / model / "tuning.json"
    sig = signature(split, rep["name"], model)
    if path.exists():
        with open(path) as fh:
            record = json.load(fh)
        if record.get("signature") == sig:
            return record["params"], record["inner_cv_MAE"]
    _, _, train, _ = next(iter_folds(split))
    t = time.perf_counter()
    params, inner_mae = run_search(model, rep, split, train, SEED)
    record = {"signature": sig, "split": split, "representation": rep["name"], "model": model,
              "tuning_fold": "repeat 0, fold 0", "n_train": int(len(train)), "seed": SEED,
              "params": to_builtin(params), "inner_cv_MAE": inner_mae,
              "search_seconds": round(time.perf_counter() - t, 1)}
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as fh:
        json.dump(record, fh, indent=1, default=str)
    log(f"    tuned [{split} | {rep['name']} | {model}] in {record['search_seconds'] / 60:.1f} min: "
          f"{ {k.replace('model__', ''): v for k, v in record['params'].items()} }")
    return record["params"], inner_mae


def fit_predict_classical(model, rep, split, train, test, seed):
    X = rep["X"]
    est = make_estimator(model, rep, seed)
    if model in PARAM_SPACES:
        if HP_MODE == "once":
            params, inner_mae = tuned_once(model, rep, split)
        else:
            params, inner_mae = run_search(model, rep, split, train, seed)
        est.set_params(**params)
        est.fit(X[train], Y[train])
        params = {k.replace("model__", ""): v for k, v in params.items()}
    else:
        est.fit(X[train], Y[train])
        params = est.fitted_params() if model == "gp" else {}
        inner_mae = np.nan
    return est.predict(X[test]), to_builtin(params), float(inner_mae)

## 6. Classical models

Each outer fold of each (split, representation, model) is stored in `results/cache/` as soon as it is
finished; a restarted run skips stored folds. The stored record carries a signature of the inputs
(file hashes, fold file, search space, `N_ITER`, `HP_MODE`, seeds, `CODE_VERSION`); records with a different
signature are recomputed. With `HP_MODE = "once"` the search result is cached as `tuning.json` next to the
fold records.

Hyperparameter searches run in a single process (RF and XGBoost use `N_JOBS` threads); between splits and
representations the GP kernel cache and joblib worker processes are released to keep memory bounded. Progress lines
are also appended, time-stamped and with the process memory, to `results/run_log.txt`. Per-fold seed: `SEED + 1000·repeat + fold`.

In [ ]:
def signature(split, rep_name, model):
    payload = {"code": CODE_VERSION, "split": split, "folds": FOLD_HASHES[split], "seed": SEED,
               "dataset": FILE_HASHES[DATASET_FILE], "conditions": FILE_HASHES[CONDITIONS_FILE],
               "model": model, "n_inner": N_INNER_FOLDS}
    if model == "chemprop":
        payload["chemprop"] = CHEMPROP
    else:
        payload["representation"] = FILE_HASHES.get(REPRESENTATIONS[rep_name][0])
        payload["space"] = PARAM_SPACES.get(model)
        payload["n_iter"] = N_ITER if model in PARAM_SPACES else None
        if model in PARAM_SPACES and HP_MODE == "once":
            payload["hp_mode"] = "once"   # absent for "per_fold" (signature of the original nested runs)
        payload["gp_restarts"] = GP_N_RESTARTS if model == "gp" else None
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def cache_path(split, rep_name, model, repeat, fold):
    return CACHE_DIR / split / rep_name / model / f"r{repeat}_f{fold:03d}.pkl"


def load_cached(path, sig):
    if not path.exists():
        return None
    with open(path, "rb") as fh:
        record = pickle.load(fh)
    return record if record.get("signature") == sig else None


def save_cached(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as fh:
        pickle.dump(record, fh)
    os.replace(tmp, path)


def run_folds(split, rep_name, model, fit_fn):
    """Runs all outer folds of one combination with fold-level caching."""
    sig = signature(split, rep_name, model)
    n_cached = n_fitted = 0
    t0 = time.perf_counter()
    for repeat, fold, train, test in iter_folds(split):
        path = cache_path(split, rep_name, model, repeat, fold)
        if load_cached(path, sig) is not None:
            n_cached += 1
            continue
        seed = SEED + 1000 * repeat + fold
        t = time.perf_counter()
        y_pred, params, inner_mae = fit_fn(split, repeat, fold, train, test, seed)
        save_cached(path, {"signature": sig, "split": split, "representation": rep_name, "model": model,
                           "repeat": repeat, "fold": fold, "reaction_id": REACTION_IDS[test],
                           "y_true": Y[test], "y_pred": np.asarray(y_pred, dtype=float).ravel(),
                           "params": params, "inner_cv_MAE": inner_mae,
                           "fit_seconds": time.perf_counter() - t, "seed": seed})
        n_fitted += 1
    log(f"  [{split} | {rep_name} | {model}] folds: {n_cached} cached, {n_fitted} fitted "
        f"({(time.perf_counter() - t0) / 60:.1f} min) | memory {memory_mb():.0f} MB")

In [ ]:
CLASSICAL_MODELS = [m for m in MODEL_ORDER if m in RUN_MODELS and m != "chemprop"]

if CLASSICAL_MODELS:
    for rep_name in RUN_REPRESENTATIONS:
        rep = load_representation(rep_name)
        if rep is None:
            log(f"{rep_name}: {REPRESENTATIONS[rep_name][0]} not found, skipped")
            continue
        n_total = rep["X"].shape[1]
        log(f"{rep_name}: {n_total} features ({rep['n_fp']} fingerprint, {n_total - rep['n_fp']} continuous)")
        for split in RUN_SPLITS:
            for model in CLASSICAL_MODELS:
                fit_fn = lambda s, r, f, tr, te, seed, m=model: fit_predict_classical(m, rep, s, tr, te, seed)
                run_folds(split, rep_name, model, fit_fn)
            release_memory()
        del rep, fit_fn
        release_memory()

## 7. chemprop D-MPNN

Two-component multicomponent MPNN:
1. `mapped_rxn` as a condensed graph of reaction (CGR, mode `REAC_DIFF`);
2. `organocatalyst` as a molecular graph (otherwise the catalyst is not seen by the model).

Extra descriptors concatenated to the learned embedding: reaction conditions and `cat_axial_R` /
`cat_axial_S` (from `axial_configuration`; atropisomeric configuration is not encoded in SMILES).
Descriptors and target are imputed/standardised with statistics of the training part of each fold.

Hyperparameters are fixed (no nested search). Early stopping and selection of the best epoch use an internal
validation part of the training data (`val_fraction`), split the same way as the outer folds:
random for `random`, by `research_group` for `lorgo_group`, by `catalyst_id` for `gkf10_catalyst`.

In [ ]:
CP = {}   # chemprop modules, featurizers and pre-parsed datapoints


def chemprop_setup():
    if CP:
        return CP
    from chemprop import data as cp_data, featurizers, models as cp_models, nn as cp_nn
    from lightning import pytorch as pl
    import torch

    for name in ("lightning", "lightning.pytorch", "lightning.fabric", "pytorch_lightning"):
        logging.getLogger(name).setLevel(logging.ERROR)
    warnings.filterwarnings("ignore", module="lightning")
    warnings.filterwarnings("ignore", message=".*num_workers.*")

    try:
        rxn_feat = featurizers.CondensedGraphOfReactionFeaturizer(mode_="REAC_DIFF")
    except TypeError:
        rxn_feat = featurizers.CondensedGraphOfReactionFeaturizer(mode="REAC_DIFF")
    if "REAC_DIFF" not in str(rxn_feat.mode).upper():
        raise RuntimeError(f"CGR mode is {rxn_feat.mode}, expected REAC_DIFF")
    mol_feat = featurizers.SimpleMoleculeMolGraphFeaturizer()

    rxn_dps, cat_dps, failed = [], [], []
    for rid, rxn, cat in zip(REACTION_IDS, data["mapped_rxn"], data["organocatalyst"]):
        try:
            rxn_dps.append(cp_data.ReactionDatapoint.from_smi(rxn))
            cat_dps.append(cp_data.MoleculeDatapoint.from_smi(cat))
        except Exception as exc:
            failed.append((rid, str(exc)[:80]))
    if failed:
        raise ValueError(f"{len(failed)} reactions could not be parsed by chemprop, e.g. {failed[:3]}")

    if CHEMPROP_DEVICE == "auto":
        accelerator = "mps" if torch.backends.mps.is_available() else "cpu"
    else:
        accelerator = CHEMPROP_DEVICE
    if accelerator == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MPS requested but not available")

    class BestWeights(pl.Callback):
        """Keeps in memory the weights of the epoch with the lowest validation loss."""

        def __init__(self):
            self.score, self.epoch, self.state = float("inf"), -1, None

        def on_validation_end(self, trainer, module):
            if trainer.sanity_checking:
                return
            value = trainer.callback_metrics.get("val_loss")
            if value is not None and float(value) < self.score:
                self.score, self.epoch = float(value), int(trainer.current_epoch)
                self.state = {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

    CP.update(data=cp_data, models=cp_models, nn=cp_nn, pl=pl, BestWeights=BestWeights,
              rxn_feat=rxn_feat, mol_feat=mol_feat,
              rxn_dps=rxn_dps, cat_dps=cat_dps, accelerator=accelerator)
    print(f"chemprop {VERSIONS['chemprop']} | accelerator: {accelerator} | {len(rxn_dps)} reactions parsed")
    return CP


def featurizer_dims(feat):
    try:
        return tuple(feat.shape)
    except AttributeError:
        return feat.atom_fdim, feat.bond_fdim


def chemprop_descriptors():
    """Conditions (merged on reaction_id) + axial flags, in dataset order."""
    return np.hstack([conditions[COND_COLS].to_numpy(float), AXIAL_FLAGS])


def fit_predict_chemprop(split, repeat, fold, train, test, seed):
    cp = chemprop_setup()
    data_, nn_, pl = cp["data"], cp["nn"], cp["pl"]
    pl.seed_everything(seed, workers=True)

    # internal validation part, grouped like the outer split
    if split == "random":
        splitter, g = ShuffleSplit(1, test_size=CHEMPROP["val_fraction"], random_state=seed), None
    else:
        splitter = GroupShuffleSplit(1, test_size=CHEMPROP["val_fraction"], random_state=seed)
        g = GROUPS[SPLIT_GROUP[split]][train]
    tr_i, va_i = next(splitter.split(np.zeros(len(train)), groups=g))
    tr, va = train[tr_i], train[va_i]

    XD = chemprop_descriptors()
    prep = BlockPreprocessor(0, scale_continuous=True).fit(XD[tr])
    mu, sd = Y[tr].mean(), Y[tr].std() or 1.0

    def dataset(idx, with_y):
        xd = prep.transform(XD[idx]).astype(np.float32)
        rxn = [replace(cp["rxn_dps"][i], y=np.array([(Y[i] - mu) / sd]) if with_y else None, x_d=xd[j])
               for j, i in enumerate(idx)]
        cat = [cp["cat_dps"][i] for i in idx]
        return data_.MulticomponentDataset([data_.ReactionDataset(rxn, cp["rxn_feat"]),
                                            data_.MoleculeDataset(cat, cp["mol_feat"])])

    c = CHEMPROP
    loaders = {
        "train": data_.build_dataloader(dataset(tr, True), batch_size=c["batch_size"], num_workers=0,
                                        seed=seed, shuffle=True),
        "val": data_.build_dataloader(dataset(va, True), batch_size=c["batch_size"], num_workers=0, shuffle=False),
        "test": data_.build_dataloader(dataset(test, False), batch_size=c["batch_size"], num_workers=0, shuffle=False),
    }

    blocks = [nn_.BondMessagePassing(*featurizer_dims(f), d_h=c["message_hidden_dim"], depth=c["depth"],
                                     dropout=c["dropout"]) for f in (cp["rxn_feat"], cp["mol_feat"])]
    mcmp = nn_.MulticomponentMessagePassing(blocks=blocks, n_components=2)
    ffn = nn_.RegressionFFN(input_dim=mcmp.output_dim + len(prep.cont_idx_),
                            hidden_dim=c["ffn_hidden_dim"], n_layers=c["ffn_num_layers"], dropout=c["dropout"])
    model = cp["models"].MulticomponentMPNN(mcmp, nn_.MeanAggregation(), ffn, batch_norm=False,
                                            warmup_epochs=c["warmup_epochs"], init_lr=c["init_lr"],
                                            max_lr=c["max_lr"], final_lr=c["final_lr"])

    best = cp["BestWeights"]()
    stop = pl.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=c["patience"])
    trainer = pl.Trainer(accelerator=cp["accelerator"], devices=1, max_epochs=c["max_epochs"],
                         callbacks=[best, stop], logger=False, enable_checkpointing=False,
                         enable_progress_bar=False, enable_model_summary=False)
    trainer.fit(model, loaders["train"], loaders["val"])
    if best.state is not None:
        model.load_state_dict(best.state)
    preds = trainer.predict(model, loaders["test"])
    y_pred = np.concatenate([p.detach().cpu().numpy().reshape(-1) for p in preds]) * sd + mu
    params = {"best_epoch": best.epoch, "epochs_trained": int(trainer.current_epoch),
              "best_val_loss": best.score, "n_train": int(len(tr)), "n_val": int(len(va))}
    return y_pred, params, np.nan

In [ ]:
if "chemprop" in RUN_MODELS:
    for split in RUN_SPLITS:
        run_folds(split, "cgr", "chemprop", fit_predict_chemprop)
        gc.collect()

## 8. Collected results

All cached combinations with a valid signature and a complete set of outer folds are collected,
including those from earlier runs with other `RUN_*` selections.

In [ ]:
records, stale = [], 0
for path in sorted(CACHE_DIR.rglob("*.pkl")):
    if path.relative_to(CACHE_DIR).parts[0] not in ALL_SPLITS:   # e.g. leave-one-catalyst-out from earlier runs
        continue
    with open(path, "rb") as fh:
        rec = pickle.load(fh)
    if rec["representation"] != "cgr" and REPRESENTATIONS[rec["representation"]][0] not in FILE_HASHES:
        stale += 1
        continue
    if rec["signature"] != signature(rec["split"], rec["representation"], rec["model"]):
        stale += 1
        continue
    records.append(rec)

by_combo = {}
for rec in records:
    by_combo.setdefault((rec["split"], rec["representation"], rec["model"]), []).append(rec)

complete, incomplete = {}, []
for key, recs in by_combo.items():
    if len(recs) == n_fold_units(key[0]):
        complete[key] = recs
    else:
        incomplete.append((*key, len(recs), n_fold_units(key[0])))

pred_rows, hp_rows = [], []
for (split, rep_name, model), recs in complete.items():
    for rec in recs:
        n = len(rec["reaction_id"])
        pred_rows.append(pd.DataFrame({"reaction_id": rec["reaction_id"], "split": split,
                                       "repeat": rec["repeat"], "fold": rec["fold"],
                                       "representation": rep_name, "model": model,
                                       "y_true": rec["y_true"], "y_pred": rec["y_pred"]}))
        hp_rows.append({"split": split, "repeat": rec["repeat"], "fold": rec["fold"],
                        "representation": rep_name, "model": model, "seed": rec["seed"],
                        "n_test": n, "inner_cv_MAE": rec["inner_cv_MAE"],
                        "fit_seconds": round(rec["fit_seconds"], 2),
                        "params": json.dumps(rec["params"], sort_keys=True, default=str)})

if not pred_rows:
    raise RuntimeError("No complete combinations in the cache")

SORT = ["split", "representation", "model", "repeat", "fold"]
PRED = pd.concat(pred_rows, ignore_index=True).sort_values(SORT + ["reaction_id"], ignore_index=True)
HPARAMS = pd.DataFrame(hp_rows).sort_values(SORT, ignore_index=True)
PRED.to_csv(RESULTS_DIR / "predictions_oof.csv", index=False)
HPARAMS.to_csv(RESULTS_DIR / "hyperparameters_by_fold.csv", index=False)

tuning_rows = []
for path in sorted(CACHE_DIR.rglob("tuning.json")):
    with open(path) as fh:
        rec = json.load(fh)
    if (rec["split"], rec["representation"], rec["model"]) in complete and HP_MODE == "once" \
            and rec["signature"] == signature(rec["split"], rec["representation"], rec["model"]):
        tuning_rows.append({k: rec[k] for k in ("split", "representation", "model", "tuning_fold", "n_train",
                                                "inner_cv_MAE", "search_seconds")}
                           | {"params": json.dumps({k.replace("model__", ""): v for k, v in rec["params"].items()},
                                                   sort_keys=True, default=str)})
if tuning_rows:
    pd.DataFrame(tuning_rows).sort_values(["split", "representation", "model"]).to_csv(
        RESULTS_DIR / "hyperparameters_tuned.csv", index=False)

print(f"Complete combinations: {len(complete)} | stale fold records ignored: {stale}")
if incomplete:
    print("Incomplete combinations (excluded):")
    for split, rep_name, model, n_have, n_need in sorted(incomplete):
        print(f"  {split} | {rep_name} | {model}: {n_have}/{n_need} folds")

## 9. Metrics

MAE, RMSE, R², Spearman ρ and Kendall τ-b are computed on pooled out-of-fold predictions (R² relative
to the mean of the evaluated reactions; rank metrics are undefined for a constant prediction). For `random`, metrics are computed per repeat and reported as mean ± sd over
the 5 repeats.

95% CIs: cluster bootstrap with `N_BOOT` replicates. Clusters are resampled with replacement and
all reactions of a sampled cluster enter with the multiplicity of that cluster; clusters are reactions
(`random`), research groups (`lorgo_group`) and catalysts (catalyst splits). For `random`, the statistic
of each replicate is the mean over repeats. The same replicates are used for all models and
representations within a split.

**Note on rank metrics under LORGO (`lorgo_group`).** Pooled ρ and τ include the shift of the training mean
between folds: when a selective research group is left out, the mean of the remaining groups is lower, so
predictions of different folds are offset against the true level of their group. Even the Dummy model obtains
ρ ≈ −0.54. Pooled rank metrics for LORGO therefore do not measure ranking quality within a research group.

In [ ]:
CLUSTER_OF_SPLIT = {"random": None, "lorgo_group": "research_group", "gkf10_catalyst": "catalyst_id"}


def bootstrap_weights(split, n_boot, seed):
    """(n_boot, n_reactions) multiplicities from resampling clusters with replacement."""
    rng = np.random.default_rng([seed, ALL_SPLITS.index(split)])
    col = CLUSTER_OF_SPLIT[split]
    codes = np.arange(len(REACTION_IDS)) if col is None else pd.factorize(GROUPS[col])[0]
    n_clusters = codes.max() + 1
    draws = rng.integers(0, n_clusters, size=(n_boot, n_clusters))
    counts = np.zeros((n_boot, n_clusters))
    np.add.at(counts, (np.repeat(np.arange(n_boot), n_clusters), draws.ravel()), 1)
    return counts[:, codes], n_clusters


def rank_stats(y, yhat):
    """Spearman rho and Kendall tau-b; NaN for a constant prediction."""
    if np.ptp(yhat) == 0:
        return np.nan, np.nan
    return spearmanr(y, yhat).statistic, kendalltau(y, yhat).statistic


def weighted_metrics(W, y, yhat):
    sw = W.sum(1)
    e = yhat - y
    mae = W @ np.abs(e) / sw
    rmse = np.sqrt(W @ e ** 2 / sw)
    ybar = W @ y / sw
    sst = W @ y ** 2 - sw * ybar ** 2
    r2 = 1.0 - (W @ e ** 2) / sst
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


BOOT = {}
alpha = (1 - CI_LEVEL) / 2
rows = []
for (split, rep_name, model), sub in PRED.groupby(["split", "representation", "model"], sort=False):
    if split not in BOOT:
        BOOT[split] = bootstrap_weights(split, N_BOOT, SEED)
    W, n_clusters = BOOT[split]
    wide = sub.pivot_table(index="reaction_id", columns="repeat", values="y_pred", aggfunc="count")
    if not (wide == 1).all().all() or len(wide) != len(REACTION_IDS):
        raise ValueError(f"{split}/{rep_name}/{model}: each reaction must be predicted once per repeat")
    pred = sub.pivot(index="reaction_id", columns="repeat", values="y_pred").loc[REACTION_IDS].to_numpy()
    y = Y
    point = {"MAE": [], "RMSE": [], "R2": []}
    boot = {"MAE": 0.0, "RMSE": 0.0, "R2": 0.0}
    for r in range(pred.shape[1]):
        point["MAE"].append(mean_absolute_error(y, pred[:, r]))
        point["RMSE"].append(np.sqrt(mean_squared_error(y, pred[:, r])))
        point["R2"].append(r2_score(y, pred[:, r]))
        for k, v in weighted_metrics(W, y, pred[:, r]).items():
            boot[k] = boot[k] + v / pred.shape[1]
    row = {"representation": rep_name, "model": model, "split": split, "n_reactions": len(y),
           "n_repeats": pred.shape[1], "n_folds": sub.groupby(["repeat", "fold"]).ngroups // pred.shape[1],
           "n_clusters": n_clusters}
    for k in ("MAE", "RMSE", "R2"):
        vals = np.asarray(point[k])
        row[k] = vals.mean()
        row[f"{k}_sd"] = vals.std(ddof=1) if len(vals) > 1 else np.nan
        row[f"{k}_ci_low"], row[f"{k}_ci_high"] = np.quantile(boot[k], [alpha, 1 - alpha])

    # rank metrics: point value per repeat; bootstrap replicate = reactions repeated by cluster multiplicity
    n_rep = pred.shape[1]
    rank_point = np.array([rank_stats(y, pred[:, r]) for r in range(n_rep)])
    rank_boot = np.zeros((N_BOOT, 2))
    for b in range(N_BOOT):
        idx = np.repeat(np.arange(len(y)), W[b].astype(int))
        rank_boot[b] = np.mean([rank_stats(y[idx], pred[idx, r]) for r in range(n_rep)], axis=0)
    for j, k in enumerate(("Spearman_rho", "Kendall_tau")):
        row[k] = np.nan if np.isnan(rank_point[:, j]).all() else np.nanmean(rank_point[:, j])
        row[f"{k}_sd"] = np.std(rank_point[:, j], ddof=1) if n_rep > 1 else np.nan
        if np.isnan(rank_boot[:, j]).all():
            row[f"{k}_ci_low"], row[f"{k}_ci_high"] = np.nan, np.nan
        else:
            row[f"{k}_ci_low"], row[f"{k}_ci_high"] = np.nanquantile(rank_boot[:, j], [alpha, 1 - alpha])
    rows.append(row)

METRICS = pd.DataFrame(rows)
METRICS["_s"] = METRICS["split"].map(ALL_SPLITS.index)
METRICS["_r"] = METRICS["representation"].map(REP_ORDER.index)
METRICS["_m"] = METRICS["model"].map(MODEL_ORDER.index)
METRICS = METRICS.sort_values(["_s", "_r", "_m"]).drop(columns=["_s", "_r", "_m"]).reset_index(drop=True)
METRICS.to_csv(RESULTS_DIR / "metrics_summary.csv", index=False, float_format="%.4f")


def fmt(row, k):
    sd = f" ± {row[f'{k}_sd']:.3f}" if pd.notna(row[f"{k}_sd"]) else ""
    return f"{row[k]:.3f}{sd} [{row[f'{k}_ci_low']:.3f}, {row[f'{k}_ci_high']:.3f}]"


for split in [s for s in ALL_SPLITS if s in METRICS["split"].unique()]:
    sub = METRICS[METRICS["split"] == split]
    table = pd.DataFrame({"Representation": sub["representation"].map(REP_LABELS),
                          "Model": sub["model"].map(MODEL_LABELS),
                          "MAE (kcal/mol)": [fmt(r, "MAE") for _, r in sub.iterrows()],
                          "RMSE (kcal/mol)": [fmt(r, "RMSE") for _, r in sub.iterrows()],
                          "R²": [fmt(r, "R2") for _, r in sub.iterrows()],
                          "Spearman ρ": [fmt(r, "Spearman_rho") for _, r in sub.iterrows()]})
    print(f"\n{SPLIT_LABELS[split]} — value{' ± sd over repeats' if split == 'random' else ''} [{int(CI_LEVEL * 100)}% CI]")
    display(table.style.hide(axis="index"))

## 10. Figures

- `parity_<split>`: observed vs predicted |ΔΔG‡| for every representation × model (for `random`,
  predictions averaged over repeats). Axes span the observed range; predictions outside it are drawn
  at the boundary and counted in the panel annotation.
- `heatmap_summary`: MAE (sequential scale, per split) and R² (diverging scale centred at 0, common to all
  splits) for representation × model.

Each figure is written as PDF and PNG (300 dpi) to `results/figures/`.

In [ ]:
MARK_COLOR = "#2a5fa8"
TEXT_COLOR = "#222222"


def save_figure(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")


def present(values, order):
    return [x for x in order if x in set(values)]


lo, hi = Y.min(), Y.max()
pad = 0.05 * (hi - lo)
LIMS = (lo - pad, hi + pad)
metric_lookup = METRICS.set_index(["split", "representation", "model"])

for split in present(PRED["split"], ALL_SPLITS):
    sub = PRED[PRED["split"] == split]
    reps = present(sub["representation"], REP_ORDER)
    models = [m for m in present(sub["model"], MODEL_ORDER) if m != "chemprop"] or ["chemprop"]
    # the chemprop panel (row "cgr") is placed in the first column
    column = lambda rep_name, model: 0 if rep_name == "cgr" else models.index(model)
    fig, axes = plt.subplots(len(reps), len(models), figsize=(1.75 * len(models) + 0.6, 1.75 * len(reps) + 0.6),
                             squeeze=False, sharex=True, sharey=True)
    used = np.zeros(axes.shape, dtype=bool)
    for (rep_name, model), d in sub.groupby(["representation", "model"]):
        i, j = reps.index(rep_name), column(rep_name, model)
        ax = axes[i, j]
        used[i, j] = True
        agg = d.groupby("reaction_id").agg(y_true=("y_true", "first"), y_pred=("y_pred", "mean"))
        outside = int(((agg["y_pred"] < LIMS[0]) | (agg["y_pred"] > LIMS[1])).sum())
        ax.plot(LIMS, LIMS, color="0.6", lw=0.8, ls="--", zorder=1)
        ax.scatter(agg["y_true"], agg["y_pred"].clip(*LIMS), s=4, alpha=0.35, color=MARK_COLOR,
                   linewidths=0, rasterized=True, zorder=2)
        m = metric_lookup.loc[(split, rep_name, model)]
        note = f"MAE {m['MAE']:.2f}\nR² {m['R2']:.2f}" + (f"\n{outside} out of range" if outside else "")
        ax.text(0.04, 0.96, note, transform=ax.transAxes, va="top", ha="left", fontsize=6.5, color=TEXT_COLOR)
        ax.set_xlim(LIMS)
        ax.set_ylim(LIMS)
        ax.set_aspect("equal")
        if rep_name == "cgr":
            ax.set_title(MODEL_LABELS[model], fontsize=8)
    for i in range(len(reps)):
        for j in range(len(models)):
            ax = axes[i, j]
            if not used[i, j]:
                ax.set_axis_off()
                continue
            if i == 0:
                ax.set_title(MODEL_LABELS[models[j]], fontsize=8)
            if j == 0:
                ax.set_ylabel(REP_LABELS[reps[i]], fontsize=8)
            if i == len(reps) - 1 or not used[i + 1, j]:   # tick labels where no panel is below / left
                ax.tick_params(labelbottom=True)
            if j == 0 or not used[i, j - 1]:
                ax.tick_params(labelleft=True)
    suffix = " — mean prediction over repeats" if split == "random" else ""
    fig.suptitle(SPLIT_LABELS[split] + suffix, fontsize=9)
    fig.tight_layout()
    fig.supxlabel("Observed |ΔΔG‡| (kcal/mol)", y=-0.01)
    fig.supylabel("Predicted |ΔΔG‡| (kcal/mol)", x=-0.015)
    save_figure(fig, f"parity_{split}")
    plt.show()

In [ ]:
def annotate_cells(ax, M, im):
    for (i, j), v in np.ndenumerate(M):
        if np.isnan(v):
            continue
        r, g, b, _ = im.cmap(im.norm(v))
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if luminance < 0.5 else TEXT_COLOR)


splits = present(METRICS["split"], ALL_SPLITS)
reps = present(METRICS["representation"], REP_ORDER)
models = present(METRICS["model"], MODEL_ORDER)
r2_abs = max(0.1, float(np.nanmax(np.abs(METRICS["R2"].clip(lower=-1)))))

fig, axes = plt.subplots(len(splits), 2, figsize=(2 * (0.55 * len(models) + 1.6), len(splits) * (0.3 * len(reps) + 0.9)),
                         squeeze=False)
for i, split in enumerate(splits):
    sub = METRICS[METRICS["split"] == split]
    for j, (metric, cmap, norm) in enumerate([
        ("MAE", "Blues", Normalize(sub["MAE"].min(), sub["MAE"].max())),
        ("R2", "RdBu", TwoSlopeNorm(vcenter=0.0, vmin=-r2_abs, vmax=r2_abs)),
    ]):
        ax = axes[i, j]
        M = (sub.pivot(index="representation", columns="model", values=metric)
                .reindex(index=reps, columns=models).to_numpy(float))
        im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, norm=norm, aspect="auto")
        ax.set_facecolor("#f2f2f2")
        annotate_cells(ax, M, im)
        ax.set_xticks(range(len(models)), [MODEL_LABELS[m] for m in models], rotation=45, ha="right")
        ax.set_yticks(range(len(reps)), [REP_LABELS[r] for r in reps])
        ax.tick_params(length=0)
        for side in ax.spines.values():
            side.set_visible(False)
        label = "MAE (kcal/mol)" if metric == "MAE" else "R²"
        ax.set_title(f"{SPLIT_LABELS[split]} — {label}", fontsize=8)
        cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
        cb.ax.tick_params(labelsize=6.5)
        cb.outline.set_visible(False)
fig.tight_layout()
save_figure(fig, "heatmap_summary")
plt.show()

## 11. Manifest

In [ ]:
manifest = {
    "notebook": "mannich_regression.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {p: package_version(p) for p in PACKAGES},
    "seeds": {"SEED": SEED, "random_repeat_seeds": RANDOM_REPEAT_SEEDS,
              "per_fold_seed": "SEED + 1000*repeat + fold", "bootstrap_seed": [SEED, "split index"]},
    "config": {"RUN_REPRESENTATIONS": RUN_REPRESENTATIONS, "RUN_MODELS": RUN_MODELS, "RUN_SPLITS": RUN_SPLITS,
               "N_FOLDS_RANDOM": N_FOLDS_RANDOM, "N_FOLDS_GKF_CATALYST": N_FOLDS_GKF_CATALYST,
               "N_INNER_FOLDS": N_INNER_FOLDS, "N_ITER": N_ITER, "HP_MODE": HP_MODE, "N_BOOT": N_BOOT, "CI_LEVEL": CI_LEVEL,
               "GP_N_RESTARTS": GP_N_RESTARTS, "CHEMPROP": CHEMPROP, "CHEMPROP_DEVICE": CHEMPROP_DEVICE,
               "PARAM_SPACES": PARAM_SPACES, "CODE_VERSION": CODE_VERSION},
    "input_files_sha256": FILE_HASHES,
    "fold_files_sha256": {f"folds_{s}.csv": h for s, h in FOLD_HASHES.items()},
    "results": {"complete_combinations": sorted("|".join(k) for k in complete),
                "incomplete_combinations": [list(x) for x in sorted(incomplete)]},
}
with open(RESULTS_DIR / "manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2, default=str)
print(f"Written: {sorted(p.name for p in RESULTS_DIR.iterdir() if p.is_file())}")